### Validamos base

In [1]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from sqlalchemy import create_engine, text

import numpy as np
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from variables_inicio import *
from funciones_spark import *
from utils_sql import *

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()


server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
server_sql = server_zeus
db_sql = "odin"
user_sql = user_zeus
pwd_sql = pwd_zeus

engine_odin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

server_sql = server_zeus
db_sql = "SAMANTHA"
user_sql = user_zeus
pwd_sql = pwd_zeus

engine_samantha = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)


engine_mysql = create_engine(
    f"mysql+pymysql://{user_envio}:{pwd_envio}@{server_envio}:{port_mysql}/{db_envio}"
)



fecha_mes_base='2026-08-01'

In [2]:
query = """
    select NUMERO_DOCUMENTO, PROPENSION_IC,frescura,cl_telf1,cl_telf2,cl_telf3,cl_telf4,
	cl_telf5,cl_telf6,cl_telf7,cl_telf8,cl_telf9,cl_telf10,cl_celular,cl_telefono   
    from DANTALION.dbo.Base_Maestra_ALFIN_BK
    where cl_base='agosto 2026'
    and cl_telf1 is not null
    and fecha_envio>='2026-08-01'
    AND RETIRO = 'ACTIVO'
    """
df_formato=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)


In [5]:
df_formato_pd=df_formato.toPandas()

In [6]:
filename='TARGET.txt'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_target_desembolso = pd.read_csv(ruta_archivo,sep='|')
df_target_desembolso = df_target_desembolso[['DNI']].copy()
df_target_desembolso['CANAL']='CANAL'
filename='ACUM_DESEM.txt'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_fugas = pd.read_csv(ruta_archivo,sep='|')
df_fugas = df_fugas[['DNI','CANALVENTA']].copy()
df_target_desembolso['DNI'] = (
    df_target_desembolso['DNI']
    .astype(str)
    .str.replace(r'\D', '', regex=True)   
    .replace('', pd.NA)                     
    .str.zfill(8)                           
)
df_fugas['DNI'] = (
    df_fugas['DNI']
    .astype(str)
    .str.replace(r'\D', '', regex=True)   
    .replace('', pd.NA)                     
    .str.zfill(8)                           
)

df_desembolso=df_fugas.merge(
    df_target_desembolso,
    on=['DNI'],
    how='left'
)
df_desembolso = df_desembolso.fillna("OTROS")
df_desembolso.rename(columns={'DNI': 'dni_cliente'}, inplace=True)


filename='RetiroDefinitivo_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_def_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_Telefonos.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_telf = pd.read_csv(ruta_archivo,sep='|')
filename='retiro_correo_alfin.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_retiro_correo = pd.read_csv(ruta_archivo,sep=';')

df_def_blacklist = df_def_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_blacklist = df_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_telf= df_telf.rename(columns={'TELEFONO': 'celular'})
df_retiro_correo= df_retiro_correo.rename(columns={'DNI': 'dni_cliente'})

df1 = df_def_blacklist.copy()
df1["celular"] = None
df1 = df1[["dni_cliente", "celular"]]

df2 = df_blacklist.copy()
df2["celular"] = None
df2 = df2[["dni_cliente", "celular"]]

df3 = df_telf.copy()
df3["dni_cliente"] = None
df3 = df3[["dni_cliente", "celular"]]

df4 = df_retiro_correo[["dni_cliente", "celular"]].copy()

df_retiros = pd.concat(
    [df1, df2, df3, df4],
    ignore_index=True
)

dni_retiro = set(df_retiros['dni_cliente'].dropna())
cel_retiro = set(df_retiros['celular'].dropna())

df_formato_pd=df_formato_pd[
    df_formato_pd['NUMERO_DOCUMENTO'].isin(dni_retiro)|
    df_formato_pd['cl_telf1'].isin(cel_retiro)|
    df_formato_pd['cl_telf2'].isin(cel_retiro)|
    df_formato_pd['cl_telf3'].isin(cel_retiro)|
    df_formato_pd['cl_telf4'].isin(cel_retiro)|
    df_formato_pd['cl_telf5'].isin(cel_retiro)|
    df_formato_pd['cl_telf6'].isin(cel_retiro)|
    df_formato_pd['cl_telf7'].isin(cel_retiro)|
    df_formato_pd['cl_telf8'].isin(cel_retiro)|
    df_formato_pd['cl_telf9'].isin(cel_retiro)|
    df_formato_pd['cl_telf10'].isin(cel_retiro)|
    df_formato_pd['cl_celular'].isin(cel_retiro)|
    df_formato_pd['cl_telefono'].isin(cel_retiro)
    ].copy()
df_formato_pd.shape


C:\Users\DATA\AppData\Local\Temp\ipykernel_6068\830441643.py:66: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_retiros = pd.concat(


(3, 15)

In [10]:
df_formato_pd["NUMERO_DOCUMENTO"] = (
    df_formato_pd["NUMERO_DOCUMENTO"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
)
df_formato_pd[['NUMERO_DOCUMENTO']].to_sql(
    name="temp_retiro_alfin",
    con=engine_kishin,
    if_exists="append",
    index=False,
    chunksize=1000
)

3

In [11]:
try:
    with engine_kishin.begin() as conn:
        query = f"""
            UPDATE a
            SET a.retiro = 'RETIRO'
            from DANTALION.dbo.Base_Maestra_ALFIN_BK a
            inner join DANTALION.dbo.temp_retiro_alfin B
            ON A.NUMERO_DOCUMENTO=B.NUMERO_DOCUMENTO
            WHERE fecha_envio >= '{fecha_mes_base}'
                AND fecha_envio <= EOMONTH('{fecha_mes_base}')
            and a.retiro='ACTIVO'
        """
        result = conn.execute(text(query))
        print("Filas actualizadas:", result.rowcount)

except Exception as e:
    print(e)

Filas actualizadas: 3


In [13]:
from sqlalchemy import text

with engine_kishin.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS temp_retiro_alfin"))
    # conn.execute(text("TRUNCATE TABLE tb_funnel_reclutamiento"))

In [35]:
query = """
    select NUMERO_DOCUMENTO as Dni, PROPENSION_IC,frescura,cl_telf1 as PHONE_NUMBER,cruce,lote   
    from DANTALION.dbo.Base_Maestra_ALFIN_BK
    where cl_telf1 is not null
    and fecha_envio>='2026-08-01'
    AND RETIRO = 'ACTIVO'
    """
df_formato=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)

query = """
	SELECT distinct Dni FROM THOTH.dbo.Tmp_LLamadas_Alfin_5 
    where Descripcion_ in(
        'EXPRESO FUTURA DENUNCIA ANTE INDECOPI O REGULADOR',
        'EXPRESO QUE NO AUTORIZÓ USO DE DATOS PERSONALES',
        'EXPRESO RECIBIR MÚLTIPLES LLAMADAS',
        'FUERA DE SERVICIO'
    )
    """
df_quitar=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)

query = f"""
    select NUMERO_DOCUMENTO as Dni,RECORRIDO from maeba.[ADM_OBJ_TG].[tGestionMesAlfin]
    WHERE FECHA_ENVIO >= '{fecha_mes_base}'
    AND FECHA_ENVIO <= EOMONTH('{fecha_mes_base}')
    """
df_maeba=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)


print(df_formato.count())
print(df_quitar.count())
print(df_maeba.count())

220294
1398
233259


In [36]:
df_desembolso = df_desembolso[
    df_desembolso["CANAL"] == "Otros"
].copy()

df_desembolso=df_desembolso.rename(columns={'dni_cliente':'Dni'})

ruta_archivo = os.path.join(ruta_csv, 'desembolso_alfin_solo_dni.csv')
df_desembolso.to_csv(ruta_archivo, sep=';')

filename='desembolso_alfin_solo_dni.csv'
df_lista_des=cargar_archivo_csv(spark,filename,';',True)

query = f"""
	SELECT dni_cliente as Dni,'1' as ref
    FROM Alice.prospectos_correos_alfin 
    where fecha_envio>='2026-08-22'
"""
df_correo_ref = pd.read_sql(query, engine_mysql)

ruta_archivo = os.path.join(ruta_csv, 'envio_correo_ref.csv')
df_correo_ref.to_csv(ruta_archivo, sep=';')

filename='envio_correo_ref.csv'
df_correo=cargar_archivo_csv(spark,filename,';',True)



In [37]:

df_formato=df_formato.join(df_quitar,['Dni'],'leftanti')
df_formato=df_formato.join(df_lista_des,['Dni'],'leftanti')
df_formato=df_formato.join(df_maeba,['Dni'],'inner')
df_formato=df_formato.join(df_correo,['Dni'],'left')
df_formato=df_formato.dropDuplicates(['Dni'])


In [38]:
df_formato=df_formato.withColumn('cruce',when(F.col('ref').isNotNull(),F.lit('CET+')).otherwise(F.col('cruce')))

In [39]:
df_formato=df_formato.filter(
    ~(
        (F.col('lote')=='NO CLIENTE')&
        (F.col('RECORRIDO')==1)
    )
    )

In [40]:
df_formato=df_formato.filter(
    ~(
        (F.col('lote')=='INVENTARIO')&
        (F.col('cruce')=='CET')&
        (F.col('ref').isNull())
    )
    )

In [42]:

df_1 = df_formato.filter(
    (F.col("lote") == "INVENTARIO") &
    (F.col("cruce") == "CET+") &
    (F.col("ref") == 1)
)


# ============================================================
# 2. INVENTARIO + NOCET + ref IS NULL
#    Queremos: 2351 * 1.7 = 3996.7 -> 3997
# ============================================================

cantidad_2 = round(2351 * 1.7)

df_2 = (
    df_formato
    .filter(
        (F.col("lote") == "INVENTARIO") &
        (F.col("cruce") == "NOCET") &
        (F.col("ref").isNull())
    )
    .orderBy(F.rand(seed=42))
    .limit(cantidad_2)
)


# ============================================================
# 3. NO CLIENTE + NULL + NULL
#    Queremos exactamente 11,023
# ============================================================

df_3 = (
    df_formato
    .filter(
        (F.col("lote") == "NO CLIENTE") &
        (F.col("cruce").isNull()) &
        (F.col("ref").isNull())
    )
    .orderBy(F.rand(seed=42))
    .limit(11023)
)


# ============================================================
# 4. NO CLIENTE + CET+ + ref = 1
#    TODOS: 86
# ============================================================

df_4 = df_formato.filter(
    (F.col("lote") == "NO CLIENTE") &
    (F.col("cruce") == "CET+") &
    (F.col("ref") == 1)
)


# ============================================================
# 5. UNIR LOS 4 GRUPOS
# ============================================================

df_final = (
    df_1
    .unionByName(df_2)
    .unionByName(df_3)
    .unionByName(df_4)
)


# ============================================================
# 6. VALIDAR RESULTADO
# ============================================================

df_final.groupBy(
    "lote",
    "cruce",
    "ref"
).count().show()

+----------+-----+----+-----+
|      lote|cruce| ref|count|
+----------+-----+----+-----+
|INVENTARIO| CET+|   1| 2351|
|INVENTARIO|NOCET|NULL| 3997|
|NO CLIENTE| NULL|NULL|11023|
|NO CLIENTE| CET+|   1|   86|
+----------+-----+----+-----+



In [ ]:
df_formato.groupBy('lote','cruce','ref') \
    .count() \
    .orderBy('lote','cruce') \
    .show(30)

+----------+-----+----+-----+
|      lote|cruce| ref|count|
+----------+-----+----+-----+
|INVENTARIO| CET+|   1| 2351|
|INVENTARIO|NOCET|NULL|39658|
|NO CLIENTE| NULL|NULL|29022|
|NO CLIENTE|  CET|NULL|  283|
|NO CLIENTE| CET+|   1|   86|
+----------+-----+----+-----+



In [29]:
df_correo.columns

['_c0', 'Dni', 'ref']

In [ ]:
df_final = df_final.withColumn(
    "cruce",
    F.when(
        F.col("ref").isNotNull(),
        F.lit("CET+")
    )
    .when(
        F.col("recorrido") == 1,
        F.concat(
            F.col("cruce"),
            F.lit("+"),
            F.lit("recorrido")
        )
    )
    .otherwise(F.col("cruce"))
)

In [ ]:
from pyspark.sql import functions as F

# ============================================================
# 1. TODOS LOS CET+
# ============================================================

df_cet = df_formato.filter(
    F.col("cruce") == "CET+"
)

# ============================================================
# 2. TODOS LOS NOCET
# Incluye NOCET y NOCET+recorrido
# ============================================================

df_nocet = df_formato.filter(
    F.col("cruce") == "NOCET+"
)

# ============================================================
# 3. 20,000 DE LOS DEMÁS QUE NO SEAN NULL
# Evitamos CET+ y NOCET para no duplicarlos
# ============================================================

df_20k = (
    df_formato
    .filter(
        F.col("cruce").isNull()
    )
    .orderBy(F.rand())
    .limit(22674)
)

# ============================================================
# 4. UNIR TODO
# ============================================================

df_formato_final = (
    df_cet
    .unionByName(df_nocet)
    .unionByName(df_20k)
)
df_formato_final=df_formato_final.dropDuplicates(['Dni'])

In [ ]:
df_formato_final.groupBy('lote','cruce') \
    .count() \
    .orderBy('lote','cruce') \
    .show(30)


## cargar recorrido

In [51]:
df_split=df_final.select( 'Dni',  'PHONE_NUMBER','PROPENSION_IC','frescura','cruce','ref')
df_split=df_split.withColumn('fecha_llamada',F.lit('2026-08-22'))
df_split=df_split.withColumn('Numero_Campana',F.lit('401'))
df_split=df_split.withColumn('list_description',F.lit('provicional'))
df_split=df_split.withColumn('list_name',F.lit('provicional'))
df_split=df_split.withColumn('Nombre_Campana',F.lit('BOT_ALFIN'))
df_split=df_split.withColumn('TIPO_LOTE',F.lit('BASE BOT'))
df_split=df_split.withColumn('DNI_Ejecutivo',F.lit('49587612')) 
df_split=df_split.withColumn('Ejecutivo',F.lit('Bot Bco Alfin')) 

df_split=df_split.withColumn('Sub_estado',F.lit(''))
df_split=df_split.withColumn('Estados',F.lit(''))

In [52]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ============================================================
# 0. CONFIGURACIÓN GENERAL
# ============================================================

FECHA_COL = "Fecha_Llamada"

# Horario permitido: 09:00 a 17:50
SEGUNDO_INICIO = (9 * 3600) + (0 * 60)
SEGUNDO_FIN = (17 * 3600) + (50 * 60)

# Bot: máximo 15 llamadas simultáneas
MAX_LLAMADAS_BOT = 5

# Separación entre llamadas
GAP_MINIMO = 15
GAP_MAXIMO = 20


# ============================================================
# 1. USUARIOS
# ============================================================

usuarios_regular = [
    "PEC200",
    "PEC188",
    "PEC136",
    "PEC139",
    "PEC184",
    "VDAD"
]

usuarios_bot = [
    "49587612",
    "VDAD"
]

regular_sin_vdad = [
    usuario
    for usuario in usuarios_regular
    if usuario != "VDAD"
]

bot_sin_vdad = [
    usuario
    for usuario in usuarios_bot
    if usuario != "VDAD"
]



# ============================================================
# 2. LIMPIAR TIPOS DE DATOS
# ============================================================

df_split_1 = (
    df_split
    .withColumn(
        FECHA_COL,
        F.to_date(F.col(FECHA_COL))
    )
    .withColumn(
        "PROPENSION_IC",
        F.col("PROPENSION_IC").cast("double")
    )
    .withColumn(
        "FRESCURA",
        F.col("FRESCURA").cast("double")
    )
)



In [53]:


# ============================================================
# 2. LIMPIAR TIPOS DE DATOS
# ============================================================

df_split_1 = (
    df_split_1
    .withColumn(
        FECHA_COL,
        F.to_date(F.col(FECHA_COL))
    )
    .withColumn(
        "PROPENSION_IC",
        F.col("PROPENSION_IC").cast("double")
    )
    .withColumn(
        "FRESCURA",
        F.col("FRESCURA").cast("double")
    )
)


# ============================================================
# 3. PROBABILIDAD DE ASIGNACIÓN A VDAD
# ============================================================

df_split_1 = df_split_1.withColumn(
    "_prob_vdad_base",
    F.when(
        F.col("TIPO_LOTE") == "BASE REGULAR",
        F.lit(0.43)
    )
    .when(
        F.col("TIPO_LOTE") == "BASE BOT",
        F.lit(0.90)
    )
    .otherwise(F.lit(0.43))
)

df_split_1 = df_split_1.withColumn(
    "_score_vdad",
    (
        (
            F.coalesce(F.col("PROPENSION_IC"), F.lit(1.0))
            - F.lit(1.0)
        ) / F.lit(5.0)
        +
        (
            F.coalesce(F.col("FRESCURA"), F.lit(0.0))
            / F.lit(5.0)
        )
    ) / F.lit(2.0)
)

df_split_1 = df_split_1.withColumn(
    "_prob_vdad_final",
    F.least(
        F.col("_prob_vdad_base")
        + F.col("_score_vdad") * F.lit(0.15),
        F.lit(0.99)
    )
)

df_split_1 = df_split_1.withColumn(
    "_rnd_vdad",
    F.rand()
)


In [54]:

# ============================================================
# 3. PROBABILIDAD DE ASIGNACIÓN A VDAD
# ============================================================

df_split_1 = df_split_1.withColumn(
    "_prob_vdad_base",
    F.when(
        F.col("TIPO_LOTE") == "BASE REGULAR",
        F.lit(0.43)
    )
    .when(
        F.col("TIPO_LOTE") == "BASE BOT",
        F.lit(0.92)
    )
    .otherwise(F.lit(0.43))
)

df_split_1 = df_split_1.withColumn(
    "_score_vdad",
    (
        (
            F.coalesce(F.col("PROPENSION_IC"), F.lit(1.0))
            - F.lit(1.0)
        ) / F.lit(5.0)
        +
        (
            F.coalesce(F.col("FRESCURA"), F.lit(0.0))
            / F.lit(5.0)
        )
    ) / F.lit(2.0)
)

df_split_1 = df_split_1.withColumn(
    "_prob_vdad_final",
    F.least(
        F.col("_prob_vdad_base")
        + F.col("_score_vdad") * F.lit(0.15),
        F.lit(0.99)
    )
)

df_split_1 = df_split_1.withColumn(
    "_rnd_vdad",
    F.rand()
)


# ============================================================
# 4. ASIGNAR DNI DEL EJECUTIVO
# ============================================================

df_split_1 = df_split_1.withColumn(
    "DNI_Ejecutivo",
    F.when(
        F.col("_rnd_vdad") <= F.col("_prob_vdad_final"),
        F.lit("VDAD")
    ).otherwise(F.lit(None).cast("string"))
)

array_regular = F.array(
    *[F.lit(usuario) for usuario in regular_sin_vdad]
)

array_bot = F.array(
    *[F.lit(usuario) for usuario in bot_sin_vdad]
)

df_split_1 = df_split_1.withColumn(
    "DNI_Ejecutivo",

    F.when(
        F.col("DNI_Ejecutivo").isNull()
        & (F.col("TIPO_LOTE") == "BASE REGULAR"),

        array_regular[
            F.floor(
                F.rand() * F.lit(len(regular_sin_vdad))
            ).cast("int")
        ]
    )

    .when(
        F.col("DNI_Ejecutivo").isNull()
        & (F.col("TIPO_LOTE") == "BASE BOT"),

        array_bot[
            F.floor(
                F.rand() * F.lit(len(bot_sin_vdad))
            ).cast("int")
        ]
    )

    .otherwise(F.col("DNI_Ejecutivo"))
)


# ============================================================
# 5. ASIGNAR NOMBRE DEL EJECUTIVO
# ============================================================

df_split_1 = df_split_1.withColumn(
    "Ejecutivo",

    F.when(
        (F.col("TIPO_LOTE") == "BASE BOT")
        & (F.col("DNI_Ejecutivo") == "VDAD"),

        F.lit("Outbound Auto Dial")
    )

    .when(
        (F.col("TIPO_LOTE") == "BASE BOT")
        & (F.col("DNI_Ejecutivo") != "VDAD"),

        F.lit("Bot Bco Alfin")
    )

    .when(
        (F.col("TIPO_LOTE") == "BASE REGULAR")
        & F.col("DNI_Ejecutivo").isin(regular_sin_vdad),

        F.lit("Agente BCO Alfin")
    )

    .when(
        (F.col("TIPO_LOTE") == "BASE REGULAR")
        & (F.col("DNI_Ejecutivo") == "VDAD"),

        F.lit("Outbound Auto Dial")
    )

    .otherwise(F.lit(None).cast("string"))
)



In [55]:

# ============================================================
# 6. CÓDIGOS DE PALETA
# ============================================================

codigos_vdad = [
    "PDROP",
    "NA",
    "AB"
]

codigos_regular = [
    "zzz5",
    "zzz6",
    "zzz9",
    "zzz13",
    "zzz10",
    "zzz11",
    "zzz56",
    "zzz23",
    "zzz24",
    "zzz26"
]

codigos_bot = [
    "zzz5",
    "zzz10",
    "zzz19",
    "zzz22",
    "zzz23",
    "zzz15",
    "zzz24",
    "zzz26"
]

array_codigos_vdad = F.array(
    *[F.lit(codigo) for codigo in codigos_vdad]
)

array_codigos_regular = F.array(
    *[F.lit(codigo) for codigo in codigos_regular]
)

array_codigos_bot = F.array(
    *[F.lit(codigo) for codigo in codigos_bot]
)


# ============================================================
# 7. ASIGNAR CÓDIGO DE PALETA
# ============================================================

df_split_1 = df_split_1.withColumn(
    "Codigo_Paleta",

    F.when(
        F.col("DNI_Ejecutivo") == "VDAD",

        array_codigos_vdad[
            F.floor(
                F.rand() * F.lit(len(codigos_vdad))
            ).cast("int")
        ]
    )

    .when(
        (F.col("TIPO_LOTE") == "BASE REGULAR")
        & (F.col("DNI_Ejecutivo") != "VDAD"),

        array_codigos_regular[
            F.floor(
                F.rand() * F.lit(len(codigos_regular))
            ).cast("int")
        ]
    )

    .when(
        (F.col("TIPO_LOTE") == "BASE BOT")
        & (F.col("DNI_Ejecutivo") != "VDAD"),

        array_codigos_bot[
            F.floor(
                F.rand() * F.lit(len(codigos_bot))
            ).cast("int")
        ]
    )

    .otherwise(F.lit(None).cast("string"))
)


# ============================================================
# 8. GENERAR DURACIÓN DE LLAMADA
# ============================================================

df_split_1 = df_split_1.withColumn(
    "segundos",

    # Llamadas sin duración
    F.when(
        F.col("Codigo_Paleta").isin(
            "PDROP",
            "AB",
            "NA",
            "zzz27",
            "zzz26",
            "zzz25",
            "zzz24",
            "zzz23"
        ),
        F.lit(0)
    )

    # Llamada larga: 100 a 150 segundos
    .when(
        F.col("Codigo_Paleta") == "zzz1",
        F.floor(
            F.rand() * F.lit(51) + F.lit(100)
        ).cast("int")
    )

    # Llamada regular: 50 a 100 segundos
    .when(
        F.col("Codigo_Paleta").isin(
            "zzz5",
            "zzz6",
            "zzz9",
            "zzz13",
            "zzz10",
            "zzz11",
            "zzz56"
        ),
        F.floor(
            F.rand() * F.lit(51) + F.lit(50)
        ).cast("int")
    )

    # Llamada BOT: 30 a 80 segundos
    .when(
        F.col("Codigo_Paleta").isin(
            "zzz19",
            "zzz15",
            "zzz22"
        ),
        F.floor(
            F.rand() * F.lit(51) + F.lit(30)
        ).cast("int")
    )

    .otherwise(F.lit(0))
)

df_split_1 = df_split_1.withColumn(
    "_duracion",
    F.coalesce(
        F.col("segundos").cast("int"),
        F.lit(0)
    )
)


# ============================================================
# 9. GENERAR IDENTIFICADOR TEMPORAL
# ============================================================

df_split_1 = df_split_1.withColumn(
    "_id_temporal",
    F.monotonically_increasing_id()
)


# ============================================================
# 10. HORARIO PARA VDAD
# ============================================================
# VDAD recibe una hora aleatoria entre 09:00 y 17:50.
# Se descuenta la duración para que no termine fuera del horario.

df_vdad = (
    df_split_1
    .filter(
        F.col("DNI_Ejecutivo") == "VDAD"
    )
    .withColumn(
        "_ultimo_inicio",
        F.greatest(
            F.lit(SEGUNDO_INICIO),
            F.lit(SEGUNDO_FIN) - F.col("_duracion")
        )
    )
    .withColumn(
        "_segundo_inicio",
        F.floor(
            F.rand()
            * (
                F.col("_ultimo_inicio")
                - F.lit(SEGUNDO_INICIO)
                + F.lit(1)
            )
            + F.lit(SEGUNDO_INICIO)
        ).cast("long")
    )
)


# ============================================================
# 11. HORARIO PARA BASE REGULAR
# ============================================================
# Las llamadas de cada agente se ordenan de manera consecutiva.
# No pueden cruzarse entre sí.

df_regular = df_split_1.filter(
    (F.col("TIPO_LOTE") == "BASE REGULAR")
    & (F.col("DNI_Ejecutivo") != "VDAD")
)

ventana_regular = (
    Window
    .partitionBy(
        FECHA_COL,
        "DNI_Ejecutivo"
    )
    .orderBy(
        F.rand()
    )
)

df_regular = df_regular.withColumn(
    "_orden",
    F.row_number().over(ventana_regular)
)

df_regular = df_regular.withColumn(
    "_gap",
    F.floor(
        F.rand() * F.lit(GAP_MAXIMO - GAP_MINIMO + 1)
        + F.lit(GAP_MINIMO)
    ).cast("int")
)

ventana_acumulada_regular = (
    Window
    .partitionBy(
        FECHA_COL,
        "DNI_Ejecutivo"
    )
    .orderBy(
        "_orden"
    )
    .rowsBetween(
        Window.unboundedPreceding,
        -1
    )
)

df_regular = df_regular.withColumn(
    "_segundo_inicio",

    F.lit(SEGUNDO_INICIO)
    +
    F.coalesce(
        F.sum(
            F.col("_duracion") + F.col("_gap")
        ).over(ventana_acumulada_regular),

        F.lit(0)
    )
)


# ============================================================
# 12. HORARIO PARA BASE BOT
# ============================================================
# Cada grupo de 15 llamadas comparte la misma hora de inicio.

df_bot = df_split_1.filter(
    (F.col("TIPO_LOTE") == "BASE BOT")
    & (F.col("DNI_Ejecutivo") != "VDAD")
)

ventana_bot = (
    Window
    .partitionBy(
        FECHA_COL,
        "DNI_Ejecutivo"
    )
    .orderBy(
        F.rand()
    )
)

df_bot = df_bot.withColumn(
    "_orden",
    F.row_number().over(ventana_bot)
)

df_bot = df_bot.withColumn(
    "_grupo_bot",
    F.floor(
        (F.col("_orden") - F.lit(1))
        / F.lit(MAX_LLAMADAS_BOT)
    ).cast("int")
)

# Duración máxima de cada grupo de 15 llamadas
df_bot_grupos = (
    df_bot
    .groupBy(
        FECHA_COL,
        "DNI_Ejecutivo",
        "_grupo_bot"
    )
    .agg(
        F.max("_duracion").alias("_duracion_grupo")
    )
    .withColumn(
        "_gap_grupo",
        F.floor(
            F.rand() * F.lit(GAP_MAXIMO - GAP_MINIMO + 1)
            + F.lit(GAP_MINIMO)
        ).cast("int")
    )
)

ventana_acumulada_bot = (
    Window
    .partitionBy(
        FECHA_COL,
        "DNI_Ejecutivo"
    )
    .orderBy(
        "_grupo_bot"
    )
    .rowsBetween(
        Window.unboundedPreceding,
        -1
    )
)

df_bot_grupos = df_bot_grupos.withColumn(
    "_segundo_inicio",

    F.lit(SEGUNDO_INICIO)
    +
    F.coalesce(
        F.sum(
            F.col("_duracion_grupo")
            + F.col("_gap_grupo")
        ).over(ventana_acumulada_bot),

        F.lit(0)
    )
)

df_bot = df_bot.join(
    df_bot_grupos.select(
        FECHA_COL,
        "DNI_Ejecutivo",
        "_grupo_bot",
        "_segundo_inicio"
    ),
    on=[
        FECHA_COL,
        "DNI_Ejecutivo",
        "_grupo_bot"
    ],
    how="left"
)


# ============================================================
# 13. UNIR REGULAR, BOT Y VDAD
# ============================================================

df_split_1 = (
    df_regular
    .unionByName(
        df_bot,
        allowMissingColumns=True
    )
    .unionByName(
        df_vdad,
        allowMissingColumns=True
    )
)


# ============================================================
# 14. VALIDAR HORARIO
# ============================================================

df_split_1 = df_split_1.withColumn(
    "_segundo_fin",
    F.col("_segundo_inicio")
    + F.col("_duracion")
)

df_split_1 = df_split_1.withColumn(
    "Dentro_Rango_Horario",

    F.when(
        (F.col("_segundo_inicio") >= F.lit(SEGUNDO_INICIO))
        &
        (F.col("_segundo_fin") <= F.lit(SEGUNDO_FIN)),

        F.lit("SI")
    )

    .otherwise(F.lit("NO"))
)


# ============================================================
# 15. GUARDAR REGISTROS FUERA DEL HORARIO
# ============================================================
# Esto permite revisar qué llamadas no alcanzaron a entrar.

df_fuera_horario = df_split_1.filter(
    F.col("Dentro_Rango_Horario") == "NO"
)


# ============================================================
# 16. CONSERVAR SOLO LLAMADAS DENTRO DEL HORARIO
# ============================================================

df_split_1 = df_split_1.filter(
    F.col("Dentro_Rango_Horario") == "SI"
)


# ============================================================
# 17. CREAR TIMESTAMP BASE DE LA FECHA
# ============================================================

df_split_1 = df_split_1.withColumn(
    "_fecha_base_timestamp",
    F.to_timestamp(
        F.date_format(
            F.col(FECHA_COL),
            "yyyy-MM-dd"
        ),
        "yyyy-MM-dd"
    )
)


# ============================================================
# 18. CREAR FECHA Y HORA DE LLAMADA
# ============================================================
# Se convierte la fecha base a segundos Unix y luego se suman
# los segundos transcurridos desde medianoche.

df_split_1 = df_split_1.withColumn(
    "Fecha_Hora_Llamada",

    F.from_unixtime(
        F.unix_timestamp(
            F.col("_fecha_base_timestamp")
        )
        +
        F.col("_segundo_inicio").cast("long")
    ).cast("timestamp")
)


# ============================================================
# 19. INICIO Y FIN DE LLAMADA
# ============================================================

df_split_1 = df_split_1.withColumn(
    "Inicio",
    F.col("Fecha_Hora_Llamada")
)

df_split_1 = df_split_1.withColumn(
    "Fin",

    F.from_unixtime(
        F.unix_timestamp(
            F.col("Inicio")
        )
        +
        F.col("segundos").cast("long")
    ).cast("timestamp")
)


# ============================================================
# 20. CREAR HORA Y TRAMA
# ============================================================

df_split_1 = df_split_1.withColumn(
    "hora",
    F.date_format(
        F.col("Inicio"),
        "HH:mm:ss"
    )
)

df_split_1 = df_split_1.withColumn(
    "Trama_Hora",
    F.hour(
        F.col("Inicio")
    )
)


# ============================================================
# 21. COLUMNAS DE ESTADO
# ============================================================

df_split_1 = df_split_1.withColumn(
    "Estados",
    F.lit("").cast("string")
)

df_split_1 = df_split_1.withColumn(
    "Sub_estado",
    F.lit("").cast("string")
)

df_split_1 = df_split_1.withColumn(
    "Codigo_Paleta",
    F.col("Codigo_Paleta").cast("string")
)


# ============================================================
# 22. LIMPIAR COLUMNAS AUXILIARES
# ============================================================

columnas_auxiliares = [
    "_prob_vdad_base",
    "_score_vdad",
    "_prob_vdad_final",
    "_rnd_vdad",
    "_duracion",
    "_id_temporal",
    "_ultimo_inicio",
    "_orden",
    "_gap",
    "_grupo_bot",
    "_duracion_grupo",
    "_gap_grupo",
    "_segundo_inicio",
    "_segundo_fin",
    "_fecha_base_timestamp"
]

columnas_a_eliminar = [
    columna
    for columna in columnas_auxiliares
    if columna in df_split_1.columns
]

df_split_1 = df_split_1.drop(
    *columnas_a_eliminar
)


# ============================================================
# 23. OPCIONAL: DEJAR UNA SOLA LLAMADA POR DNI
# ============================================================
# Descomenta únicamente si necesitas un registro por cliente.

# df_split_1 = df_split_1.dropDuplicates(["Dni"])


# ============================================================
# 24. ORDENAR RESULTADO
# ============================================================

df_split_1 = df_split_1.orderBy(
    F.col("Fecha_Hora_Llamada").asc(),
    F.col("DNI_Ejecutivo").asc()
)



In [56]:

df_split_1 = df_split_1.withColumn(
    "Trama_Hora",
    F.hour(F.col("Inicio"))
)
df_split_1 = df_split_1.withColumn(
    "Codigo_Paleta",
    F.col("Codigo_Paleta").cast("string")
)

In [57]:
segundos_random = (
    F.floor(F.rand() * (91 - 60 + 1)) + 60
)


In [58]:

df_split_1=df_split_1.withColumn('Codigo_Paleta',when(
        (F.col("ref")==1)
    ,'zzz2').otherwise(F.col('Codigo_Paleta'))) 

In [59]:
df_split_1 = df_split_1.withColumn(
    "Fin",
    F.when(
        (F.col("ref")==1)
        ,
        F.from_unixtime(
            F.unix_timestamp("Inicio") + segundos_random
        ).cast("timestamp")
    ).otherwise(F.col("Fin"))
)

In [60]:
df_split_1=df_split_1.withColumn('segundos',
    when(
        (F.col("ref")==1)
        ,F.unix_timestamp(F.col("Fin")) - F.unix_timestamp(F.col("Inicio")))
    .otherwise(F.col('segundos'))
) 


In [ ]:
df_split_2=df_split_1.select('Dni', 'DNI_Ejecutivo', 'Ejecutivo', 'Fecha_Hora_Llamada', 'segundos', 'Fecha_Llamada', 'Trama_Hora', 'PHONE_NUMBER', 'Codigo_Paleta', 'Inicio', 'Fin','Numero_Campana','list_name','list_description','Nombre_Campana','Sub_estado','Estados')


In [62]:
df_split_2=df_split_2.dropDuplicates(['Dni'])
df_split_2.count()

17457

In [64]:
df_split_2=df_split_2.withColumn('fecha_llamada',F.lit('2026-08-22'))
# df_split_2.count()

In [65]:
df_split_2=df_split_2.dropDuplicates(['Dni'])

In [67]:
df_split_2=df_split_2.withColumn('Sub_estado',F.lit(''))
df_split_2=df_split_2.withColumn('Estados',F.lit(''))

In [68]:
append_table_SQL(spark,df_split_2,f'tmp_llamadas_mes',server_zeus,user_zeus,pwd_zeus,'SAMANTHA')


### actualiar temp llamadas zzz por vacio

In [69]:
try:
    with engine_samantha.begin() as conn:
        query = f"""
            update SAMANTHA..tmp_llamadas_mes
            set Codigo_Paleta=REPLACE(Codigo_Paleta,'zzz','')
            where Numero_Campana='401'
            and Fecha_Llamada>='2026-08-22'
        """
        result = conn.execute(text(query))
        print("Filas actualizadas:", result.rowcount)

except Exception as e:
    print(e)

Filas actualizadas: 17457


## armar feedback

In [74]:
exec_query_sql(server_zeus, "THOTH", user_zeus, pwd_zeus, "EXEC Cronox.Generador_Gestion_Mes_Alfin", "SP actualizar llamadas vicidial Zeus")


SP actualizar llamadas vicidial Zeus | realizado | duración: 22.5 seg


In [75]:
query = """
select 
'TAG202' as COD_CANAL,
'TARGET' as CANAL,
a.Dni as DNI,
'20260801' as FECHA_ENVIO,
FORMAT(a.Fecha_Llamada, 'yyyyMMdd') AS FECHA_GESTION,
CONVERT(VARCHAR(8), a.Fecha_Hora_Llamada, 108) AS HORA_GESTION,
a.PHONE_NUMBER as TELEFONO,
'T1' as ORIGEN_TELEFONO,
b.campania as COD_CAMPANA,
a.Codigo_paleta  as COD_TIPO,
b.oferta_max as OFERTA,
'' as DNI_ASESOR
,fecha_llamada
from THOTH.dbo.Tmp_LLamadas_Alfin a
inner join ODIN.dbo.Base_Maestra_Alfin_bk_Vigente b
on a.Dni=b.NUMERO_DOCUMENTO
where a.fecha_llamada>='2026-08-22'
    """
print(df_formato1.columns)
df_formato1=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)
print(df_formato1.count())


['COD_CANAL', 'CANAL', 'DNI', 'FECHA_ENVIO', 'FECHA_GESTION', 'HORA_GESTION', 'TELEFONO', 'ORIGEN_TELEFONO', 'COD_CAMPANA', 'COD_TIPO', 'OFERTA', 'DNI_ASESOR', 'fecha_llamada']
18124


In [76]:
print([row['COD_TIPO' ] for row in df_formato1.select('COD_TIPO').distinct().collect()])


['7', '15', '3', '8', '22', '16', '5', '18', '27', '17', '26', '6', '19', '23', '24', '9', '10', '12', '2']


### generar feedback

In [77]:
df_feedback_pd = df_formato1.toPandas()


In [78]:

# Convertir la columna de fecha
df_feedback_pd["FECHA_INICIO_GESTION_ref"] = pd.to_datetime(
    df_feedback_pd["fecha_llamada"],
    errors="coerce"
)

# Crear una columna auxiliar solo con la fecha
df_feedback_pd["fecha_archivo"] = (
    df_feedback_pd["FECHA_INICIO_GESTION_ref"].dt.date
)

# Eliminar registros cuya fecha no pudo convertirse
df_feedback_pd = df_feedback_pd[
    df_feedback_pd["fecha_archivo"].notna()
]

# Crear la carpeta si no existe
os.makedirs(ruta_csv, exist_ok=True)

# Generar un archivo CSV por cada fecha
for fecha, df_dia in df_feedback_pd.groupby("fecha_archivo"):
    nombre_fecha = fecha.strftime("%Y%m%d")
    nombre_archivo = f"aFEEDBACK_TARGET_{nombre_fecha}.csv"
    # nombre_fecha = fecha.strftime("%d_%m_%Y")
    # nombre_archivo = f"FEEDBACK_TARGET_{nombre_fecha}.csv"

    ruta_archivo = os.path.join(
        ruta_csv,
        nombre_archivo
    )

    # Quitar columnas auxiliares antes de exportar
    df_exportar = df_dia.drop(
        columns=[
            "FECHA_INICIO_GESTION_ref",
            "fecha_archivo",
            'fecha_llamada'
        ]
    )

    # Exportar CSV separado por |
    df_exportar.to_csv(
        ruta_archivo,
        sep="|",
        index=False,
        encoding="utf-8-sig",
        lineterminator="\n"
    )

    print(
        f"Archivo generado: {ruta_archivo} "
        f"| filas: {len(df_exportar)}"
    )

Archivo generado: C:\Users\DATA\Documents\datos\05_subir_csv\aFEEDBACK_TARGET_20260822.csv | filas: 18124


In [139]:
query = """
    select top(1)* from SAMANTHA..tmp_llamadas_mes
    where Numero_Campana='401'
    and Fecha_Llamada>='2026-08-21'
    """
df_111=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)
print(df_111.columns)

['Dni', 'Numero_Campana', 'Nombre_Campana', 'DNI_Ejecutivo', 'Ejecutivo', 'Fecha_Hora_Llamada', 'segundos', 'Fecha_Llamada', 'Trama_Hora', 'Estados', 'Sub_estado', 'Descripcion', 'list_description', 'list_name', 'PHONE_NUMBER', 'Fecha_Agenda', 'Comentarios', 'Codigo_Paleta', 'Inicio', 'Fin', 'lead_id']


In [136]:

# Convertir la columna de fecha
df_feedback_pd["FECHA_INICIO_GESTION_ref"] = pd.to_datetime(
    df_feedback_pd["fecha_llamada"],
    errors="coerce"
)

# Crear una columna auxiliar solo con la fecha
df_feedback_pd["fecha_archivo"] = (
    df_feedback_pd["FECHA_INICIO_GESTION_ref"].dt.date
)

# Eliminar registros cuya fecha no pudo convertirse
df_feedback_pd = df_feedback_pd[
    df_feedback_pd["fecha_archivo"].notna()
]

# Crear la carpeta si no existe
os.makedirs(ruta_csv, exist_ok=True)

# Generar un archivo CSV por cada fecha
for fecha, df_dia in df_feedback_pd.groupby("fecha_archivo"):
    nombre_fecha = fecha.strftime("%Y%m%d")
    nombre_archivo = f"aFEEDBACK_TARGET_{nombre_fecha}.csv"
    # nombre_fecha = fecha.strftime("%d_%m_%Y")
    # nombre_archivo = f"FEEDBACK_TARGET_{nombre_fecha}.csv"

    ruta_archivo = os.path.join(
        ruta_csv,
        nombre_archivo
    )

    # Quitar columnas auxiliares antes de exportar
    df_exportar = df_dia.drop(
        columns=[
            "FECHA_INICIO_GESTION_ref",
            "fecha_archivo",
            'fecha_llamada'
        ]
    )

    # Exportar CSV separado por |
    df_exportar.to_csv(
        ruta_archivo,
        sep="|",
        index=False,
        encoding="utf-8-sig",
        lineterminator="\n"
    )

    print(
        f"Archivo generado: {ruta_archivo} "
        f"| filas: {len(df_exportar)}"
    )

NameError: name 'df_feedback_pd' is not defined

In [1]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from sqlalchemy import create_engine
from sqlalchemy import text

import numpy as np

server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

engine_mysql = create_engine(
    f"mysql+pymysql://{user_envio}:{pwd_envio}@{server_envio}:{port_mysql}/{db_envio}"
)


In [2]:
query = f"""
	select * from Alice.prospectos_correos_alfin
"""
df_correo = pd.read_sql(query, engine_mysql)


query = f"""
	select * from Alice.prospectos_envio_alfin
"""
df_formulario = pd.read_sql(query, engine_mysql)

In [5]:
df_correo[df_correo['fecha_visita']=='2026-07-11'].head()

,id,canal_campo,supervisor,ejecutivo_target,codigo_ejecutivo_id,cdv_alfin_banco,dni_cliente,nombre_cliente,color,monto_solicitado,celular,agencia_atencion,fecha_visita,hora_visita,fecha_envio,intentos_realizados,estado,tipo_carga,fecha_registro,fecha_dia


In [17]:
query = f"""
	select NUMERO_DOCUMENTO as dni_cliente,color_final as color from DANTALION.dbo.Base_Maestra_Alfin_bk_Vigente
"""
df_maestra = pd.read_sql(query, engine_kishin)

df_maestra["dni_cliente"] = (
    df_maestra["dni_cliente"]
    .astype(str)
    .str.zfill(8)
)
print(df_maestra.columns.tolist())


['dni_cliente', 'color']


In [18]:
filename='fomato_agendas_alfin_credicash_2026.xlsx'
filePath = os.path.join(ruta_csv, filename)
df_formato = pd.read_excel(filePath)
df_formato['supervisor']='CARLOS ENRIQUE RAMIREZ CACHIQUE'
df_formato['canal_campo']='CALL CENTER / TARGET OUTSOURCING'
df_formato['codigo_ejecutivo_id']='00000001'
df_formato['ejecutivo_target']='BOT'
df_formato['cdv_alfin_banco']='ROSA HONOR'

df_formato = df_formato.rename(columns={
    'monto': 'monto_solicitado'
})
df_formato["dni_cliente"] = (
    df_formato["dni_cliente"]
    .astype(str)
    .str.zfill(8)
)
df_formato = df_formato.merge(
    df_maestra,
    on='dni_cliente',
    how='left'
)

df_formato['fecha_visita']='2026-07-10'

# Semilla opcional para reproducibilidad
# np.random.seed(123)

# Horas posibles: 09 a 18
horas = np.random.randint(9, 19, size=len(df_formato))

# Minutos posibles
minutos = np.random.choice([0, 15, 30, 45], size=len(df_formato))

# Crear la columna
df_formato["hora_visita"] = [
    f"{h:02d}:{m:02d}:00"
    for h, m in zip(horas, minutos)
]

df_formato['telefono_cliente']=df_formato['celular']
df_formato['dni_vendedor']=df_formato['ejecutivo_target']
df_formato['agencia_tienda']=df_formato['cod_agencia']
df_formato['operador']='TARGET'
df_formato['tipo_gestion']='Derivacion'

In [19]:
equivalencias = {
    'SAN JUAN DE LURIG': 'SAN JUAN DE LURIGANCHO',
    'ENMANCIPACION': 'EMANCIPACION',
    'PC HUANCAYO': 'HUANCAYO',
    'PC TACNA': 'TACNA',
    'PC HUARAZ': 'HUARAZ',
    'TRUJ CENTRO': 'TRUJILLO CENTRO',
    'TRUJ AMERICA': 'TRUJILLO AMERICA',
    'AREQ CAYMA': 'AREQUIPA CAYMA',
    'AREQ PAMPILLA': 'AREQUIPA PAMPILLA'
}

df_formato['agencia_atencion'] = (
    df_formato['agencia_atencion']
    .replace(equivalencias)
)
# df_formato.drop_duplicates(subset=["dni"], inplace=True)

In [20]:
df_formato.drop_duplicates(subset=["dni_cliente"], inplace=True)



In [21]:
df_formato.shape

(1135, 19)

In [20]:
# df_formato = df_formato[
#     df_formato['agencia_atencion'].isin([
#         'SAN JUAN DE LURIG',
#         'ENMANCIPACION',
#         'PC HUANCAYO',
#         'TRUJ CENTRO',
#         'PC TACNA',
#         'PC HUARAZ',
#         'TRUJ AMERICA',
#         'AREQ CAYMA',
#         'AREQ PAMPILLA'
#     ])
# ]

#### Validar el nombre de la agencia

In [22]:
query = f"""
	select * from Alice.agencias_alfin
"""
df_agencia = pd.read_sql(query, engine_mysql)

set_correo = set(
    df_formato['agencia_atencion']
    .dropna()
    .drop_duplicates()
)

set_agencia = set(
    df_agencia['agencia_correo']
    .dropna()
    .drop_duplicates()
)
# print(set_correo & set_agencia)
print(set_agencia - set_correo)
print(set_correo -set_agencia )

{'ICA'}
set()


In [23]:
print(set_agencia - set_correo)
print(set_correo -set_agencia )

{'ICA'}
set()


#### validar el codigo de agencia 

In [24]:

set_correo = set(
    df_formato['agencia_tienda']
    .dropna()
    .drop_duplicates()
)

set_agencia = set(
    df_agencia['agencia_Formulario']
    .dropna()
    .drop_duplicates()
)
# print(set_correo & set_agencia)
print(set_agencia - set_correo)
print(set_correo -set_agencia )

{'730879 - PAITA', '739580 - ICA'}
set()


In [26]:
df_correo['tipo_carga']='MANUAL'

In [25]:
df_correo=df_formato[['canal_campo', 'supervisor', 'ejecutivo_target', 'codigo_ejecutivo_id', 'cdv_alfin_banco', 'dni_cliente', 'nombre_cliente', 'color', 'monto_solicitado', 'celular', 'agencia_atencion', 'fecha_visita','hora_visita']] .copy()

df_formulario=df_formato[['dni_vendedor', 'operador', 'dni_cliente', 'nombre_cliente', 'telefono_cliente', 'agencia_tienda', 'fecha_visita', 'monto_solicitado', 'tipo_gestion']].copy()

display(df_correo.head(2))
display(df_formulario.head(2))


,canal_campo,supervisor,ejecutivo_target,codigo_ejecutivo_id,cdv_alfin_banco,dni_cliente,nombre_cliente,color,monto_solicitado,celular,agencia_atencion,fecha_visita,hora_visita
0,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,70286844,FRANK ALEXIS NOLAZCO CUZCANO,NaN,2800,965739962,CAÑETE,2026-07-10,13:45:00
1,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,71590543,CRISTOPHER EDDIN MANUEL CHUMAN FARROÑAN,NaN,2900,994413738,MOSHOQUEQUE,2026-07-10,18:00:00


,dni_vendedor,operador,dni_cliente,nombre_cliente,telefono_cliente,agencia_tienda,fecha_visita,monto_solicitado,tipo_gestion
0,BOT,TARGET,70286844,FRANK ALEXIS NOLAZCO CUZCANO,965739962,732243 - CAÑETE,2026-07-10,2800,Derivacion
1,BOT,TARGET,71590543,CRISTOPHER EDDIN MANUEL CHUMAN FARROÑAN,994413738,738360 - MOSHOQUEQUE,2026-07-10,2900,Derivacion


In [27]:

df_correo.to_sql(
    name="prospectos_correos_alfin",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

df_formulario.to_sql(
    name="prospectos_envio_alfin",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

1135

,canal_campo,supervisor,ejecutivo_target,codigo_ejecutivo_id,cdv_alfin_banco,dni_cliente,nombre_cliente,color,monto_solicitado,celular,agencia_atencion,fecha_visita,hora_visita
0,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,00000001,BOT,ROSA HONOR,09750804,AMADO VILLALTA,NaN,23000,942707381,CASTILLA,2026-07-08,14:45:00
1,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,00000001,BOT,ROSA HONOR,08931784,LUIS GUILLERMO CHUQUIJAJAS,NaN,23000,950955962,VILLA EL SALVADOR 2,2026-07-08,14:30:00


,dni_vendedor,operador,dni_cliente,nombre_cliente,telefono_cliente,agencia_tienda,fecha_visita,monto_solicitado,tipo_gestion
0,00000001,TARGET,09750804,AMADO VILLALTA,942707381,737490 - CASTILLA,2026-07-08,23000,Derivacion
1,00000001,TARGET,08931784,LUIS GUILLERMO CHUQUIJAJAS,950955962,732249 - VILLA EL SALVADOR 2,2026-07-08,23000,Derivacion


## Otros

In [18]:
import pandas as pd

# ============================================================
# MAPEO CODIGO_PALETA -> CODIGO
# ============================================================

map_cod_tipo = {
    "1": "1",
    "2": "2",
    "3": "3",
    "4": "4",
    "5": "5",
    "6": "6",
    "7": "7",
    "8": "8",
    "9": "9",
    "10": "10",
    "11": "11",
    "12": "12",
    "13": "13",
    "14": "14",
    "15": "15",
    "16": "16",
    "17": "17",
    "18": "18",
    "19": "19",
    "20": "20",
    "21": "21",
    "22": "22",
    "23": "23",
    "24": "24",
    "25": "25",
    "26": "26",
    "27": "27",
    "zzz15": "15",
    "CHS012": "12",
    "zzz23": "23",
    "zzz22": "22",
    "zzz19": "19",
    "zzz5": "5",
    "zzz10": "10",
    "zzz24": "24",
    "zzz26": "26",
    "CHS005": "3",
    "CHS004": "4",
    "CHS010": "10",
    "CHS022": "22",
    "CHS018": "18",
    "CHS009": "9",
    "CHS014": "14",
    "CHS023": "3",
    "CHS015": "15",
    "CHS011": "11",
    "CHS007": "7",
    "CHS001": "3",
    "CHS008": "8",
    "CHS006": "6",
    "CHS019": "19",
    "CHS020": "20",
    "CHS016": "16",
    "CHS024": "24",
    "CHS003": "3",
    "zzz2": "2",
    "NA": "27",
    "AB": "27",
    "PDROP": "27",

    # Casos adicionales de Codigo_paleta
    "DISPO": "27",
    "DCMX": "27",
    "XFER": "27",
    "INCALL": "27",
    "DONEM": "27",
    "32": "24"
}

# ============================================================
# CONVERTIR DICCIONARIO A DATAFRAME
# ============================================================

df_codigos = pd.DataFrame(
    list(map_cod_tipo.items()),
    columns=["Codigo_paleta", "codigo"]
)
df_codigos = (
    df_codigos
    .drop_duplicates(subset=["Codigo_paleta"])
    [["Codigo_paleta", "codigo"]]
    .reset_index(drop=True)
)
df_codigos.head(2)

,Codigo_paleta,codigo
0,1,1
1,2,2


In [19]:
df_codigos.to_sql(
    name="codigo_paleta_alfin_actual",
    con=engine_odin,
    if_exists="replace",
    index=False,
    chunksize=1000
)



65

In [21]:
query = """
    select * from [THOTH].dbo.Tmp_LLamadas_Alfin
    """
df_tipi=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)
print(df_tipi.columns)


['Dni', 'Numero_Campana', 'Nombre_Campana', 'DNI_Ejecutivo', 'Ejecutivo', 'Fecha_Hora_Llamada', 'segundos', 'Fecha_Llamada', 'Trama_Hora', 'Estados', 'Sub_estado', 'Descripcion', 'list_description', 'list_name', 'PHONE_NUMBER', 'Fecha_Agenda', 'Comentarios', 'Codigo_Paleta', 'Inicio', 'Fin', 'lead_id', 'Estado_', 'Sub_Estado_', 'Descripcion_', 'Pesos', 'Enlace', 'Fecha_Llam', 'Hora_Llamada', 'RH', 'COD_BCO']


In [ ]:
['Dni', 'Numero_Campana', 'Nombre_Campana', 'DNI_Ejecutivo', 'Ejecutivo', 'Fecha_Hora_Llamada', 'segundos', 'Fecha_Llamada', 'Trama_Hora', 'Estados', 'Sub_estado', 'Descripcion', 'list_description', 'list_name', 'PHONE_NUMBER', 'Fecha_Agenda', 'Comentarios', 'Codigo_Paleta', 'Inicio', 'Fin', 'lead_id']pale

In [22]:
print([row['Codigo_Paleta' ] for row in df_tipi.select('Codigo_Paleta').distinct().collect()])


['7', '15', '11', '3', '8', '22', '16', '5', '18', '27', '17', '26', '6', '19', '23', '25', '9', '24', '1', '20', '10', '4', '12', '13', '14', '21', '2']


In [ ]:
['7', '15', '11', '3', '8', '22', '16', '5', '18', '27', '17', '26', '6', '19', '23', '25', '9', '24', '1', '20', '10', '4', '12', '13', '14', '21', '2']

In [ ]:
['CHS004', 'PDROP', 'CHS005', '7', '15', 'AM', '3', 'CHS016', '8', 'CHS010', 'CHS012', '16', 'NA', 'CHS015', 'DISPO', 'CHS001', 'CHS007', '5', '18', 'CHS011', '17', '26', 'DROP', 'CHS024', 'CHS022', '6', '19', '23', 'AL', 'INCALL', 'CHS014', '25', 'CHS006', 'CHS020', 'CHS009', 'AB', 'ADC', '24', '9', '1', 'DCMX', '10', 'CHS003', 'CHS023', '4', '12', '13', 'CHS019', '14', '21', '2', 'CHS008', 'CHS018']

In [ ]:
['CHS004', 'PDROP', 'CHS005', '7', '15', 'AM', '3', 'CHS016', '8', 'CHS010', '22', 'CHS012', '16', 'NA', 'CHS015', 'DISPO', 'CHS001', 'CHS007', '5', '18', 'CHS011', '17', '26', 'DROP', 'CHS024', 'CHS022', '6', '19', '23', 'AL', 'INCALL', 'CHS014', '25', 'CHS006', 'CHS020', 'CHS009', 'AB', 'ADC', '24', '9', '1', '10', 'DCMX', 'CHS003', 'CHS023', '4', '12', '13', 'CHS019', '14', '21', '2', 'CHS008', 'CHS018']